# Forecasting

**Perishable Demand Forecasting & Zero-Waste Inventory Engine** - stage 2 of 4.

Notebook 01 filled in the demand that stockouts hid. This notebook forecasts it.

| Step | What it does |
|---|---|
| 1 | load the recovered demand from notebook 01 |
| 2 | search hyperparameters, then train the best one |
| 3 | compare against the baselines |
| 4 | train the same model on raw sales and compare the two |

A **range**, not a number: the model outputs `q10`/`q50`/`q90`, because ordering perishables means
knowing the downside as well as the middle. The test week is never touched here.

Runs on CPU, but the GPU runtime is far faster (Colab: *Runtime -> Change runtime type -> GPU*).

## Setup

In [ ]:
import warnings

import pandas as pd
import torch
warnings.filterwarnings("ignore")   # pytorch-forecasting is noisy about dataloader workers

from src.utils import config, data_io
from src.utils.metrics import quantile_scores
from src import forecast, recovery

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)"
print(f"torch {torch.__version__} | GPU: {gpu}")

## 1. Load the recovered demand

In [ ]:
daily = recovery.load_daily().merge(
    data_io.load("recovered")[["store_id", "product_id", "dt", "recovered_demand"]],
    on=["store_id", "product_id", "dt"], how="left")

assert daily["recovered_demand"].notna().all(), "run notebook 01 step 4 first"
print(f"{len(daily):,} rows | recovery model:", recovery.load_params()["model"])
daily.groupby("period")[["sale_amount", "recovered_demand"]].mean().round(4)

## 2. Hyperparameter search

The settings are searched, not guessed. `forecast.GRID` has five settings and **all 48 combinations
are scored**.

| setting | what it controls |
|---|---|
| `learning_rate` | step size. Too high and it never settles; too low and it stops before learning |
| `hidden_size` | model capacity. Bigger fits more shapes, and overfits 62 days of history sooner |
| `encoder_days` | history read per forecast. Longer sees more seasonality but gives fewer training windows |
| `target_transform` | how the target is scaled per product. `log1p` compresses the big days so the zero-heavy majority is not drowned out |
| `dropout` | regularisation - randomly drops connections while training |

Each combination stops at 10 epochs and is scored on **validation** - enough to separate them, not
to finish a fit. The best one is retrained in full below.

**Scored on `pinball(avg)`, not WAPE.** WAPE only grades the middle guess; ordering uses the whole
q10/q50/q90 range, so a model with a good middle and a useless range must not win.

**Resumes by itself** - the CSV is rewritten after each combination and ones already in it are
skipped. If Colab disconnects, run the cell again.

Roughly 45-70 minutes on a GPU. If a winning value sits at the edge of its list, widen that list in
`forecast.GRID` and re-run.

In [ ]:
TUNE = True   # False = load the saved ranking; re-run after a disconnect to resume

tuning = (forecast.tune(daily, max_epochs=10) if TUNE
          else pd.read_csv(config.tft_tuning("recovered")))
tuning.drop(columns="config_id").round(4)

### Train the best settings

The search runs stop at 10 epochs, so the winner is retrained from scratch with the full epoch
budget and early stopping.

| | |
|---|---|
| **Target** | `recovered_demand` (raw `sale_amount` in step 4) |
| **History** | the winning `encoder_days`, read by the model itself - no hand-built lag columns |
| **Known ahead** | day of week, discount, holiday, activity, weather |
| **Per product** | store, product and the three category IDs, as learned embeddings |
| **Horizon** | 7 days, rolled forward across the period |

Each 7-day block is forecast from history that stops the day before it starts, so no block sees
inside itself. Early-stopped on validation, and the best epoch is saved.

In [ ]:
BEST = forecast.best_params(tuning)
print("winning config:", BEST)

TRAIN   = True
PERIODS = ("validation", "calibration")

if TRAIN:
    recovered = forecast.run(daily, tag="recovered", max_epochs=30, **BEST)
else:
    recovered = {p: pd.read_parquet(config.forecast_parquet(p, "recovered")) for p in PERIODS}

recovered["validation"][["dt", "store_id", "product_id",
                         "q10", "q50", "q90", "sale_amount", "is_censored"]].head()

## 3. Compare against the baselines

Same scoring as notebook 01: **per date, on non-stockout rows only**, against recorded
`sale_amount` - which keeps these numbers comparable to the baseline scorecard.

- **WAPE / MAE** - how far off the middle guess (`q50`) is. Lower is better.
- **WPE** - direction. Positive over-forecasts, negative under-forecasts.
- **pinball(avg) / CRPS~** - whether the whole range is right, not just the middle. This is what the
  ordering stage actually uses.

In [ ]:
baselines = pd.read_csv(config.BASELINE_SCORECARD, index_col=0)

pd.concat([pd.DataFrame({"tft_recovered": quantile_scores(recovered["validation"])}).T,
           baselines]).round(4)

## 4. Recovered demand vs raw sales

The same settings, seed and features, trained on raw `sale_amount` instead of `recovered_demand`.
Only the target changes.

**This table cannot settle which is better.** Scoring skips stockout rows, and stockout rows are the
only rows recovery changes - on the rows being scored, the two targets are identical. So the raw
version predicts exactly what it is measured against, while the recovered version predicts *demand*
and is marked down for exceeding recorded sales.

What this table can show is the weaker but necessary point: that recovery has not made forecasting
worse. Whether it helps is measured at the ordering stage, in waste and stockouts.

In [ ]:
# the SAME winning settings: only the target changes
if TRAIN:
    raw = forecast.run(daily, tag="raw", max_epochs=30, **BEST)
else:
    raw = {p: pd.read_parquet(config.forecast_parquet(p, "raw")) for p in PERIODS}

both = {"tft_recovered": recovered["validation"], "tft_raw": raw["validation"]}
scorecard = pd.concat([pd.DataFrame({k: quantile_scores(v) for k, v in both.items()}).T, baselines])
scorecard.to_csv(config.FORECAST_SCORECARD)
scorecard.round(4)

### Forecast gap on stockout days

The scoring above hides the difference; the forecasts themselves show it. On days that ran out, the
recovered version should predict noticeably more than the raw one - and on days that did not, the two
should almost agree. That gap is the demand the raw model never learned existed.

In [ ]:
keys = ["store_id", "product_id", "dt"]
gap = (recovered["validation"][keys + ["is_censored", "q50"]]
       .rename(columns={"q50": "q50_recovered"})
       .merge(raw["validation"][keys + ["q50"]].rename(columns={"q50": "q50_raw"}), on=keys))

(gap.groupby("is_censored")[["q50_raw", "q50_recovered"]].mean()
    .assign(gap_pct=lambda d: (d.q50_recovered / d.q50_raw - 1) * 100).round(4))